# 2. PIM and Conditional Access — Implementation

SC-900 taught you *what* PIM and Conditional Access are. AZ-500 expects you to *configure* them.

## PIM implementation

### PIM settings you must configure

For each role, PIM has configurable settings:

| Setting | Options | Exam default |
|---------|---------|---------------|
| **Maximum activation duration** | 0.5 – 24 hours | 8 hours |
| **Require MFA on activation** | Yes/No | Yes (for privileged roles) |
| **Require justification** | Yes/No | Yes |
| **Require approval** | Yes/No + who approves | Yes for Global Admin |
| **Require ticket info** | Yes/No | No |
| **Allow permanent eligible** | Yes/No | No (set expiration) |
| **Allow permanent active** | Yes/No | No |
| **Notification on activation** | Email to admins | Yes |

In [ ]:
import json
from datetime import datetime, timedelta

# PIM role settings for a real-world configuration
PIM_ROLE_SETTINGS = {
    'Global Administrator': {
        'max_activation_hours': 2,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': True,
        'approvers': ['security-team@contoso.com'],
        'eligible_max_months': 6,
        'permanent_eligible': False,
        'notifications': ['admins@contoso.com'],
    },
    'Contributor': {
        'max_activation_hours': 8,
        'require_mfa': True,
        'require_justification': True,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': False,
        'notifications': [],
    },
    'Security Reader': {
        'max_activation_hours': 8,
        'require_mfa': False,
        'require_justification': False,
        'require_approval': False,
        'approvers': [],
        'eligible_max_months': 12,
        'permanent_eligible': True,
        'notifications': [],
    },
}

# Simulate PIM activation workflow
def activate_pim_role(user: str, role: str, justification: str, mfa: bool, ticket: str = '') -> dict:
    settings = PIM_ROLE_SETTINGS.get(role)
    if not settings:
        return {'status': 'error', 'reason': f'Role {role} not found in PIM'}
    
    checks = []
    if settings['require_mfa']:
        checks.append(('MFA', mfa, 'MFA is required for this role'))
    if settings['require_justification']:
        checks.append(('Justification', bool(justification), 'Justification is required'))
    if settings['require_approval']:
        checks.append(('Approval', False, f'Pending approval from {settings["approvers"]}'))
    
    failed = [c for c in checks if not c[1]]
    now = datetime.now()
    
    if failed and not settings['require_approval']:
        return {'status': '❌ DENIED', 'failed_checks': [{'check': c[0], 'reason': c[2]} for c in failed]}
    
    if settings['require_approval']:
        return {
            'status': '⏳ PENDING APPROVAL',
            'role': role,
            'user': user,
            'justification': justification,
            'approvers': settings['approvers'],
            'will_expire': (now + timedelta(hours=settings['max_activation_hours'])).strftime('%Y-%m-%d %H:%M'),
            'mfa_verified': mfa,
        }
    
    return {
        'status': '✅ ACTIVATED',
        'role': role,
        'user': user,
        'active_until': (now + timedelta(hours=settings['max_activation_hours'])).strftime('%Y-%m-%d %H:%M'),
    }

print('=== PIM Activation Scenarios ===\n')

print('--- 1. Global Admin (requires approval) ---')
print(json.dumps(activate_pim_role('alice', 'Global Administrator', 'Emergency CA policy fix', True), indent=2))

print('\n--- 2. Contributor (no approval, just MFA + justification) ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix to prod', True), indent=2))

print('\n--- 3. Contributor without MFA ---')
print(json.dumps(activate_pim_role('bob', 'Contributor', 'Deploy hotfix', False), indent=2))

### Azure CLI for PIM

```bash
# List eligible role assignments
az role assignment list --assignee alice@contoso.com --include-inherited

# PIM is managed via the Microsoft Graph API or PowerShell:
# Activate a role
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/roleManagement/directory/roleAssignmentScheduleRequests' \
  --body '{
    "action": "selfActivate",
    "justification": "Emergency CA policy fix",
    "roleDefinitionId": "<role-id>",
    "directoryScopeId": "/",
    "principalId": "<user-id>",
    "scheduleInfo": {
      "startDateTime": "2026-04-17T10:00:00Z",
      "expiration": {"type": "afterDuration", "duration": "PT2H"}
    }
  }'
```

---
## Conditional Access — implementation details

### Policy structure

Every CA policy has three parts:
1. **Assignments** — who (users/groups), what (apps), where (conditions)
2. **Access controls** — grant/block + requirements (MFA, compliant device, etc.)
3. **Session controls** — sign-in frequency, persistent browser, app-enforced restrictions

In [ ]:
# Common CA policies every AZ-500 engineer should know how to create

CA_POLICIES = [
    {
        'name': 'Require MFA for admins',
        'state': 'enabled',
        'assignments': {
            'users': {'include_roles': ['Global Administrator', 'Security Administrator', 'Exchange Administrator']},
            'apps': {'include': 'all'},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa']},
        'scenario': 'Baseline: admins always need MFA.',
    },
    {
        'name': 'Block legacy authentication',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all'},
            'apps': {'include': 'all'},
            'conditions': {'client_apps': ['exchangeActiveSync', 'other']},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Legacy protocols (IMAP, POP3, SMTP) cannot do MFA. Block them.',
    },
    {
        'name': 'Require compliant device for Azure portal',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all'},
            'apps': {'include': ['Microsoft Azure Management']},
        },
        'grant': {'operator': 'AND', 'controls': ['compliant_device']},
        'scenario': 'Only managed devices can access the Azure portal.',
    },
    {
        'name': 'Require MFA for risky sign-ins',
        'state': 'enabled',
        'assignments': {
            'users': {'include': 'all'},
            'apps': {'include': 'all'},
            'conditions': {'sign_in_risk': ['medium', 'high']},
        },
        'grant': {'operator': 'AND', 'controls': ['mfa']},
        'scenario': 'Identity Protection feeds risk signals. Medium/high risk → force MFA.',
    },
    {
        'name': 'Block access from untrusted countries',
        'state': 'report-only',
        'assignments': {
            'users': {'include': 'all'},
            'apps': {'include': 'all'},
            'conditions': {'locations': {'include': 'all', 'exclude': ['named:TrustedCountries']}},
        },
        'grant': {'operator': 'OR', 'controls': ['block']},
        'scenario': 'Start in report-only mode, then enforce after reviewing logs.',
    },
]

print('=== Conditional Access Policies to Implement ===\n')
for p in CA_POLICIES:
    state_icon = {'enabled': '🟢', 'report-only': '🟡', 'disabled': '⬜'}[p['state']]
    print(f'{state_icon} {p["name"]} [{p["state"]}]')
    print(f'   Scenario: {p["scenario"]}')
    print(f'   Grant controls: {p["grant"]["controls"]}')
    print()

### Named locations

Before using location-based policies, create named locations:

```bash
# Create a named location for trusted office IPs
az rest --method POST \
  --uri 'https://graph.microsoft.com/v1.0/identity/conditionalAccess/namedLocations' \
  --body '{
    "@odata.type": "#microsoft.graph.ipNamedLocation",
    "displayName": "Corporate offices",
    "isTrusted": true,
    "ipRanges": [
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "203.0.113.0/24"},
      {"@odata.type": "#microsoft.graph.iPv4CidrRange", "cidrAddress": "198.51.100.0/24"}
    ]
  }'
```

### Report-only mode

**Always start new policies in report-only mode.** This logs what *would* happen without actually blocking anyone. Review the sign-in logs, then switch to enabled.

### Exam tip

- CA policies are **additive** — most restrictive wins.
- You cannot use CA to grant access that isn't already permitted by RBAC.
- **Emergency access accounts** (break-glass) should be excluded from all CA policies.
- Requires Entra ID **Premium P1** minimum.

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **PIM settings** | Max activation duration, MFA, justification, approval, notifications |
| **PIM scope** | Works for Entra roles and Azure resource roles |
| **CA policy structure** | Assignments (who/what/where) → Grant controls → Session controls |
| **Named locations** | Define trusted IPs before using location-based policies |
| **Report-only mode** | Always test before enforcing |
| **Break-glass accounts** | Exclude from all CA policies |

**Next**: [Notebook 3 — App Registrations and Managed Identities](03_app_registrations_and_managed_identities.ipynb)